# Capdyssey: 첫 PPG 심방세동 분류 모델

**위에서 아래로 셀을 실행하세요.** 데이터 다운로드 → 사람별 분할 → 작은 CNN 학습 → 시험 결과 → 파일 저장 순서입니다.

1. 상단 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다. GPU가 없어도 CPU로 실행됩니다.
2. **파일 → Drive에 사본 저장**으로 본인 노트북을 보관합니다.
3. 아래 셀을 차례로 실행합니다. Drive 연결을 선택하면 계정 승인이 필요합니다.

MIMIC PERform AF는 35명의 침상 PPG이며 손목·운동 자료가 아닙니다. AF 라벨은 원본 기록의 상태입니다. 새로운 사람별 약 60/20/20 분할을 적용합니다. 기본 20 epoch, 검증 손실이 5회 개선되지 않으면 종료합니다. 시험셋은 마지막에 평가하며 임계값은 0.5로 고정합니다.

이 노트북은 연구용 기초 모델입니다. BUT 품질 모델·IMU 융합·실시간 센서 연결은 포함되지 않습니다. 시험 결과를 보고 분할 seed를 반복 선택하지 마세요.

[데이터·이용 조건](https://github.com/dung-ho03/capdyssey/blob/main/docs/DATA_LICENSES.md) · [원본 출처](https://zenodo.org/records/15906524)

**학습 완료된 연구용 v1도 저장소에 있습니다.** 오탐이 많아 서비스 배포용이 아닙니다. [실측 결과와 한계](https://github.com/dung-ho03/capdyssey/blob/main/docs/AF_MODEL_V1.md)를 읽어주세요.


## 1. 팀 코드와 실행 환경 준비

In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path('/content/capdyssey')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/dung-ho03/capdyssey.git', str(REPO)], check=True)
else:
    print('기존 폴더를 사용합니다. 최신 코드가 필요하면 새 런타임에서 시작하세요.')
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-train.txt'], check=True)
import torch
print('학습 장치:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('코드 버전:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. 결과 저장 위치 설정
기본값은 본인의 Drive입니다. `SAVE_TO_DRIVE=False`이면 임시 Colab 저장소를 사용하므로 마지막 ZIP을 반드시 내려받으세요. 각 실행은 새 폴더에 저장되어 이전 실험을 덮어쓰지 않습니다.

In [ ]:
from datetime import datetime, timezone
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    result_root = Path('/content/drive/MyDrive/Capdyssey/runs')
else:
    result_root = REPO / 'runs'
RUN = result_root / ('af_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN.parent.mkdir(parents=True, exist_ok=True)
print('결과 저장 예정:', RUN)


## 3. 공식 데이터 다운로드 및 해시 검사
약 38MB의 원본 MAT 2개를 받습니다. 원본은 임시 런타임에 저장됩니다. 연결 종료 후 새 런타임에서는 다시 받을 수 있습니다. 해시가 다르면 학습을 진행하지 않습니다.

In [ ]:
subprocess.run([sys.executable, 'scripts/download_data.py', 'mimic_af'], check=True)


## 4. 실제 파형 확인
두 파형의 모습만으로 진단하는 단계가 아닙니다. 데이터가 정상적으로 읽히는지 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(REPO / 'scripts'))
from train_af import load_records
records = load_records(REPO / 'data/raw/mimic_af')
print('참가자:', len(records), '| AF:', sum(r[1] for r in records))
fig, axes = plt.subplots(2, 1, figsize=(12, 5))
for label, ax in enumerate(axes):
    person, _, values = next(r for r in records if r[1] == label)
    ax.plot(np.arange(1250) / 125, values[:1250])
    ax.set_title(f'{person}: ' + ('AF' if label else 'Non-AF'))
    ax.set_xlabel('seconds')
plt.tight_layout()


## 5. 첫 모델 학습
먼저 기본 설정으로 실행하세요. 30초 비중첩 구간을 50Hz로 변환하고, 각 구간 안에서 정규화합니다. 결측·거의 평평한 구간만 제외하므로 정교한 품질 판별은 아닙니다. 클래스 가중치는 학습 자료에서만 계산합니다. 검증셋으로 최적 epoch를 선택하고 시험셋은 최종 평가에 사용합니다.

Drive 저장을 켰으면 최적 모델과 학습 기록이 매 epoch 저장됩니다. 런타임 종료 후 자동 이어학습 기능은 아직 없으며, 재실행은 새 실험입니다.

In [ ]:
EPOCHS = 20
SEED = 42
BATCH_SIZE = 64
subprocess.run([
    sys.executable, '-u', 'scripts/train_af.py',
    '--epochs', str(EPOCHS), '--seed', str(SEED),
    '--batch-size', str(BATCH_SIZE), '--output', str(RUN)
], check=True)


## 6. 결과 읽기
민감도는 AF를 잡은 비율, 특이도는 비AF를 올바르게 구분한 비율입니다. FN은 놓친 AF, FP는 잘못 알린 비AF입니다. 시험 참가자가 적으므로 이 결과만으로 실제 손목 센서나 임상 성능을 주장할 수 없습니다. 같은 사람의 창들은 독립된 사람이 아닙니다.

In [ ]:
import json
result = json.loads((RUN / 'metrics.json').read_text())
print(json.dumps(result, indent=2, ensure_ascii=False))
history = json.loads((RUN / 'history.json').read_text())
plt.figure(figsize=(8, 3))
plt.plot([x['epoch'] for x in history], [x['train_loss'] for x in history], label='weighted train loss')
plt.plot([x['epoch'] for x in history], [x['val_loss'] for x in history], label='validation loss')
plt.xlabel('epoch'); plt.legend(); plt.show()
m = result['test_window_metrics']
matrix = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(matrix, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, str(matrix[i,j]), ha='center', va='center')
ax.set_xticks([0,1], ['Non-AF','AF']); ax.set_yticks([0,1], ['Non-AF','AF'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Reference'); plt.show()


## 7. 결과 ZIP 내려받기
모델·평가표·시험 예측·분할 목록·전처리 설정·학습 기록이 저장됩니다. 이 ZIP은 원본 파형을 포함하지 않습니다. 공유 시에도 데이터 출처와 이용 조건을 유지하세요.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/' + RUN.name, 'zip', RUN)
print('모델:', RUN / 'best_model.pt')
files.download(archive)


## 8. 저장한 모델로 한 구간 예측
학습과 동일한 전처리를 사용합니다. 아래는 이미 평가한 시험 구간을 다시 읽는 **동작 시연**이며 새 성능 검증이 아닙니다.
다른 데이터는 30초 길이의 1차원 원본 PPG 배열과 실제 샘플링 주파수를 전달하세요. 손목 센서에서의 정확도는 검증되지 않았습니다. af_score는 진단 확률이 아닙니다.


In [ ]:
from predict_af import classify
split = json.loads((RUN / 'split.json').read_text())
person, label, raw = next(r for r in records if r[0] in split['test'])
from train_af import preprocess_window
for start in range(0, len(raw) - 3750 + 1, 3750):
    try:
        preprocess_window(raw[start:start+3750], 125)
    except ValueError:
        continue
    prediction = classify(RUN / 'best_model.pt', raw[start:start+3750], 125)
    print('원본 기록의 정답:', 'AF' if label else 'non-AF')
    print(json.dumps(prediction, indent=2, ensure_ascii=False))
    break
else:
    raise ValueError('예측 가능한 구간이 없습니다.')


## 학습 없이 v1 사용하기 (선택)
1번 환경 준비 셀만 실행한 뒤 다음 코드를 사용하면 학습을 반복하지 않아도 됩니다. 원본 데이터의 개인 PC 다운로드도 필요 없습니다.
`my_ppg`는 팀이 준비한 30초 원본 PPG 배열, `actual_sampling_hz`는 해당 센서의 실제 초당 샘플 수입니다.

```python
sys.path.insert(0, str(REPO / 'scripts'))
from predict_af import classify
result = classify(REPO / 'models/af_mimic_v1/best_model.pt', my_ppg, actual_sampling_hz)
print(result)
```
결측·평평한 파형·길이 오류는 예외로 처리됩니다. 이것은 별도의 품질 AI를 대신하지 않습니다.
